# 04 - Rationale Revision (item 8)

Local/deterministic sentence-removal evaluation from `rationale_revision.py`, plus the ready-to-run (needs `OPENROUTER_API_KEY`) evidence-constrained regeneration template.

In [ ]:
# == Environment + paths =========================================================
import json, os, re, sys, time
from pathlib import Path

# Kaggle detection uses an env var Kaggle actually sets, not path existence: on
# Windows, Path('/kaggle/working').exists() is NOT a reliable off-Kaggle check --
# pathlib treats a leading '/' as relative to the current drive's root, so an
# unrelated local C:\kaggle or F:\kaggle directory would silently return True.
ON_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or os.environ.get("KAGGLE_URL_BASE"))

def _find(name, prefer_largest=True):
    """Search for `name`; if multiple matches exist, prefer the largest file --
    a complete/authoritative cache is virtually always bigger than a stale or
    partial one left over from an earlier run."""
    roots = ["/kaggle/input"] if ON_KAGGLE else ["..", "../..", "."]
    hits = []
    for root in roots:
        r = Path(root)
        if r.exists():
            hits.extend(r.rglob(name))
    if not hits:
        return None
    if prefer_largest:
        hits.sort(key=lambda p: p.stat().st_size, reverse=True)
    return hits[0]

def _find_src_dir():
    """Locate the directory holding this project's shared modules (identified by
    containing graph_utils.py, not by a fixed relative path) -- on Kaggle, an
    attached dataset lands under /kaggle/input/<dataset-name>/..., so neither
    '../src' nor 'src' exists there; search by content the same way _find() above
    locates data files, instead of assuming a fixed layout."""
    for c in (Path("../src"), Path("src"), Path("/kaggle/working/src")):
        if (c / "graph_utils.py").exists():
            return c
    if ON_KAGGLE:
        hits = list(Path("/kaggle/input").rglob("graph_utils.py"))
        if hits:
            return hits[0].parent
    return Path("src")  # last resort; will raise ModuleNotFoundError on import if wrong

SRC_DIR = _find_src_dir()
sys.path.insert(0, str(SRC_DIR))

BENCH = _find("scireqa_full_relational_v1.1_humanmerged.jsonl") or _find("scireqa_full_relational_v1.1.jsonl")
REFS_CACHE = _find("citecheck_refs_cache.json")
CITE_RESULTS = _find("citecheck_results.json")
BENCH_VERIFIED = _find("scireqa_human_verified_subset.jsonl")
assert BENCH and REFS_CACHE and CITE_RESULTS, "attach the SciRecBench + CiteCheck output dataset"

OUT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path("../outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ON_KAGGLE:", ON_KAGGLE)
print("SRC_DIR:", SRC_DIR.resolve() if SRC_DIR.exists() else f"{SRC_DIR} (NOT FOUND -- attach the src/ dataset)")
print("BENCH:", BENCH, f"({BENCH.stat().st_size:,} bytes)" if BENCH else "")
print("REFS_CACHE:", REFS_CACHE, f"({REFS_CACHE.stat().st_size:,} bytes)" if REFS_CACHE else "")
print("CITE_RESULTS:", CITE_RESULTS)
print("BENCH_VERIFIED:", BENCH_VERIFIED)
print("OUT_DIR:", OUT_DIR)


In [ ]:
import json
from graph_utils import CitationGraph
from rationale_revision import evaluate_corpus, build_regeneration_prompt, flagged_relational_sentences, revise_by_sentence_removal

graph = CitationGraph.load(REFS_CACHE)

def load_jsonl(p):
    return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]

full_corpus = load_jsonl(BENCH)
verified_68 = load_jsonl(BENCH_VERIFIED) if BENCH_VERIFIED else []
print(f"Full corpus: {len(full_corpus)} | Human-verified: {len(verified_68)}")

results = {"full_corpus_10000": evaluate_corpus(full_corpus, graph)}
if verified_68:
    results["human_verified_68"] = evaluate_corpus(verified_68, graph)

for name, r in results.items():
    print(f"\n--- {name} ---")
    for k, v in r.items():
        print(f"  {k}: {v}")

with open(OUT_DIR / "rationale_revision_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
print("\nSaved to", OUT_DIR / "rationale_revision_results.json")


## Evidence-constrained regeneration (needs `OPENROUTER_API_KEY`)

Not run automatically -- costs API credit per call. Generates a rewrite prompt per flagged instance; wire up the same OpenRouter call pattern as `../notebooks/06_critic_generator_agent.ipynb` to actually execute it.

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    OPENROUTER_API_KEY = UserSecretsClient().get_secret("OPENROUTER_API_KEY")
except Exception:
    OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
print("OpenRouter key present:", bool(OPENROUTER_API_KEY))

sample_prompts = []
for inst in full_corpus[:5]:
    rationale = (inst.get("reference_rationale") or "").strip()
    if not rationale:
        continue
    flagged = flagged_relational_sentences(inst, graph)
    if not flagged:
        continue
    kept = [s for s in rationale.split(". ") if s not in flagged]
    sample_prompts.append(build_regeneration_prompt(rationale, flagged, kept))

print(f"Built {len(sample_prompts)} sample regeneration prompts (not sent -- template only).")
if sample_prompts:
    print("\n--- example prompt ---\n")
    print(sample_prompts[0][:800])
